In [ ]:
pip install selenium requests beautifulsoup4


# LOR1_LOR2

source activate myenv

pip install ipykernel

python -m ipykernel install --user --name myenv --display-name "Python (myenv)"

https://www.aemo.com.au/market-notices?marketNoticeQuery=&fromdate=&todate=&marketNoticeFacets=LOR1+ACTUAL

https://www.aemo.com.au/market-notices?marketNoticeQuery=&fromdate=&todate=&marketNoticeFacets=LOR2+ACTUAL
- all data in: https://nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/

In [ ]:
from selenium.webdriver.support.ui import Select
import time
import os
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service

In [ ]:


# Set the path to your ChromeDriver executable
CHROMEDRIVER_PATH = "chromedriver.exe"  # Replace with the actual path

# URL of the AEMO data page
BASE_URL = "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/"

# Folder to save the downloaded CSV files
OUTPUT_FOLDER = "NEM_market_notice_CSV_Data"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Initialize Selenium WebDriver with the Service class
service = Service(CHROMEDRIVER_PATH)
driver = webdriver.Chrome(service=service)

# Open the AEMO page
driver.get(BASE_URL)



In [ ]:
# //a[contains(@href,"MMSDM/2")]
urls_d = driver.find_elements(By.XPATH, '//a[contains(@href,"MMSDM/2")]') 

In [ ]:
urls = []
for url in urls_d:
    urls.append(url.get_attribute("href"))

In [ ]:
len(urls),urls[0]

In [ ]:
# //a[contains(@href,"MMSDM/2") and not(contains(@href, '.zip'))]
sub_urls = []
for ur in urls:
    driver.get(ur)
    urls_d = driver.find_elements(By.XPATH, '//a[contains(@href,"MMSDM/2") and not(contains(@href, ".zip"))]') 
    for url in urls_d:
        sub_urls.append(url.get_attribute("href"))

In [ ]:
sub_urls

In [ ]:
#//a[contains(@href,"MMSDM_Historical_Data_SQLLoader")]
sub_sub_urls = []
for sub_url in sub_urls:
    driver.get(sub_url)
    urls_d = driver.find_element(By.XPATH, '//a[contains(@href,"MMSDM_Historical_Data_SQLLoader")]') 
    sub_sub_urls.append(urls_d.get_attribute("href"))

In [ ]:
sub_sub_urls

In [ ]:
#//a[contains(@href,"MMSDM_Historical_Data_SQLLoader/DATA")]
sub_sub_sub_urls = []
for sub_sub_url in sub_sub_urls:
    driver.get(sub_sub_url)
    urls_d = driver.find_element(By.XPATH, '//a[contains(@href,"MMSDM_Historical_Data_SQLLoader/DATA")]') 
    sub_sub_sub_urls.append(urls_d.get_attribute("href"))

In [ ]:
sub_sub_sub_urls

In [ ]:
#//a[contains(@href,"MMSDM_Historical_Data_SQLLoader/DATA/PUBLIC_DVD_DISPATCHPRICE")]
#//a[contains(@href,"DISPATCHPRICE") and not(contains(@href,"PREDISPATCHPRICE"))]
#//a[contains(@href,"MARKETNOTICEDATA") and not(contains(@href,"DMP"))]
final_urls = []
for sub_sub_sub_url in sub_sub_sub_urls:
    driver.get(sub_sub_sub_url)
    try:
        urls_d = driver.find_element(By.XPATH, '//a[contains(@href,"MARKETNOTICEDATA") and not(contains(@href,"DMP"))]')
        final_urls.append(urls_d.get_attribute("href"))
    except:
        pass

In [ ]:
final_urls

In [ ]:
import os
import requests

In [ ]:
# Function to download and save files
def download_csv_files(links, folder):
    for link in links:
        try:
            # Get the file name from the link
            filename = link.split("/")[-1]
            filepath = os.path.join(folder, filename)
            
            # Download the file
            response = requests.get(link, stream=True)
            response.raise_for_status()  # Raise exception for HTTP errors
            
            # Save the file
            with open(filepath, "wb") as file:
                for chunk in response.iter_content(chunk_size=1024):
                    file.write(chunk)
            print(f"Downloaded: {filename}")
        
        except Exception as e:
            print(f"Failed to download {link}: {e}")

# Call the function to download CSV files
download_csv_files(final_urls, OUTPUT_FOLDER)

print("All downloads completed.")

In [ ]:
iframe = driver.find_element(By.XPATH, '//iframe[@class="has-height"]')  # Replace with actual iframe ID
driver.switch_to.frame(iframe)

In [ ]:
# Locate the dropdown elements
area_dropdown = Select(driver.find_element(By.XPATH,'//select[@id="region-select"]'))  # Replace 'area' with the actual ID
year_dropdown = Select(driver.find_element(By.XPATH,'//select[@id="year-select"]'))  # Replace 'year' with the actual ID
month_dropdown = Select(driver.find_element(By.XPATH,'//select[@id="month-select"]'))  # Replace 'month' with the actual ID

In [ ]:


# List to store the links
links = []

# Iterate through all combinations of the dropdowns
for area_option in area_dropdown.options:
    area_dropdown.select_by_visible_text(area_option.text)
    for year_option in year_dropdown.options:
        year_dropdown.select_by_visible_text(year_option.text)
        for month_option in month_dropdown.options:
            month_dropdown.select_by_visible_text(month_option.text)
            
            # Allow the page to load (adjust sleep as needed)
            time.sleep(3)
            
            # Locate the link inside the same <div> using XPath
            try:
                link_element = driver.find_element(By.XPATH, '//div[@class="form-spacing"]//a[contains(text(), "DOWNLOAD HISTORICAL DATA")]')  # Adjust XPath to match the link location
                link = link_element.get_attribute("href")
                links.append(link)
                print(f"Combination: Area={area_option.text}, Year={year_option.text}, Month={month_option.text} -> Link: {link}")
            except Exception as e:
                print(f"No link found for Area={area_option.text}, Year={year_option.text}, Month={month_option.text}")

In [ ]:
len(links)

In [ ]:
import os
import requests


# Folder to save the downloaded CSV files
output_folder = "Downloaded_CSVs"
os.makedirs(output_folder, exist_ok=True)  # Create folder if it doesn't exist

# Function to download and save files
def download_csv_files(links, folder):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    for link in links:
        try:
            # Get the file name from the link
            filename = link.split("/")[-1]
            filepath = os.path.join(folder, filename)
            
            # Download the file with headers
            response = requests.get(link, headers=headers, stream=True)
            response.raise_for_status()  # Raise exception for HTTP errors
            
            # Save the file
            with open(filepath, "wb") as file:
                for chunk in response.iter_content(chunk_size=1024):
                    file.write(chunk)
            print(f"Downloaded: {filename}")
        
        except Exception as e:
            print(f"Failed to download {link}: {e}")

# Call the function to download CSV files
download_csv_files(links, output_folder)

print("All downloads completed.")


In [ ]:


# Folder to save the downloaded CSV files
output_folder = "Downloaded_CSVs"
os.makedirs(output_folder, exist_ok=True)  # Create folder if it doesn't exist

# Function to download and save files
def download_csv_files(links, folder):
    for link in links:
        try:
            # Get the file name from the link
            filename = link.split("/")[-1]
            filepath = os.path.join(folder, filename)
            
            # Download the file
            response = requests.get(link, stream=True)
            response.raise_for_status()  # Raise exception for HTTP errors
            
            # Save the file
            with open(filepath, "wb") as file:
                for chunk in response.iter_content(chunk_size=1024):
                    file.write(chunk)
            print(f"Downloaded: {filename}")
        
        except Exception as e:
            print(f"Failed to download {link}: {e}")

# Call the function to download CSV files
download_csv_files(links, output_folder)

print("All downloads completed.")


In [ ]:
# Locate the <select> element by its ID
dropdown = driver.find_element(By.ID, "dropdown")

In [ ]:
# Wait for the page to load completely (adjust time if necessary)
driver.implicitly_wait(10)

# Function to download a file
def download_file(url, folder):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Check for HTTP errors
        filename = url.split("/")[-1]
        filepath = os.path.join(folder, filename)
        with open(filepath, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024):
                file.write(chunk)
        print(f"Downloaded: {filename}")
    except Exception as e:
        print(f"Failed to download {url}: {e}")

# Locate all links on the page
links = driver.find_elements(By.TAG_NAME, "a")

# Process and download all CSV files
for link in links:
    href = link.get_attribute("href")
    if href and href.endswith(".csv"):  # Only consider CSV files
        print(f"Found CSV: {href}")
        download_file(href, OUTPUT_FOLDER)

# Close the Selenium driver
driver.quit()

print("All downloads completed.")


# Scrapping generator data